In [1]:
import time
import logging
from pathlib import Path

from sqlalchemy.orm import exc

from debug_logging_utils import configure_loggers

In [2]:
logs_dir = Path("logs")
logs_dir.mkdir(parents=True, exist_ok=True)
app_logger = logging.getLogger("app")
log_configs = configure_loggers(filename="logger_config.yaml")

In [3]:
# Will be exluded both from console and file handler because LogRecord was never created by the app_logger
app_logger.debug("will not be logged")
# The log record will be created but the console will excude it due to the CustomFilter
app_logger.info("this is an info message", extra={"include": False})
# Will be logged from both handlers
app_logger.warning("This is a warning message")

# Will be logged only in the file
try:
    raise ValueError("This is a test exception")
except ValueError:
    app_logger.exception("An exception occurred",
                         stack_info=True,
                         extra={"include": False})

app - WARNING - This is a warning message


In [4]:
for _ in range(100):
    app_logger.info("this is an info message", extra={"include": False})
    time.sleep(0.1)


### Why and How to Configure the SQLAlchemy Logger

Configuring the SQLAlchemy logger this way allows us to:

- **Filter only important queries** (`SELECT`, `INSERT`, `UPDATE`, `DELETE`, `CREATE`, `ALTER`).
- **Format exceptions in JSON**, capturing the query, parameters, and error summary for easier debugging.

This reduces noise and makes logs more readable and actionable.In this section we will try to configure the default logger from the SQLAlchemy library. The main reason of doing this is because I consider the default logger to verbose.

In [30]:
import importlib
import logging
import sqlalchemy as sa
from sqlalchemy.exc import IntegrityError
import debug_logging_utils

# Reload the debug logging module in case it has been updated during the session
importlib.reload(debug_logging_utils)

# Create an in-memory SQLite database for demonstration
engine = sa.create_engine("sqlite:///:memory:", echo=True)

# Get the default SQLAlchemy logger
sa_logger = logging.getLogger("sqlalchemy")

# Configure SQLAlchemy logging using a custom YAML config
sa_log_config = debug_logging_utils.configure_loggers(
    filename="sqlalchemy_log_config.yaml"
)

# Start a connection context with the SQLite database
with engine.connect() as conn:

    # --------------------------
    # Step 1: Create a table
    # --------------------------
    conn.execute(sa.text("""
    CREATE TABLE IF NOT EXISTS users (
        id INTEGER PRIMARY KEY,
        username TEXT
    )
    """))

    # --------------------------
    # Step 2: Insert initial rows
    # --------------------------
    # Insert two users: Alice and Bob
    # Note: SQLite automatically assigns id because it's a primary key
    conn.execute(sa.text("INSERT INTO users (username) VALUES (:name)"), {"name": "Alice"})
    conn.execute(sa.text("INSERT INTO users (username) VALUES (:name)"), {"name": "Bob"})

    # Commit changes (required for SQLite when using a raw connection)
    conn.commit()

    # --------------------------
    # Step 3: Demonstrate IntegrityError
    # --------------------------
    # Attempting to insert a row with a duplicate primary key (id=1)
    # This triggers an IntegrityError due to UNIQUE constraint violation
    try:
        conn.execute(sa.text("INSERT INTO users (id, username) VALUES (:id, :name)"), {"id": 1, "name": "Bob"})
    except IntegrityError as e:
        # Log the exception using the configured SQLAlchemy logger
        # `exc_info=True` ensures the full exception info is included
        sa_logger.error(e, exc_info=True)

    # --------------------------
    # Step 4: Query the table
    # --------------------------
    # Retrieve all users to verify current data
    result = conn.execute(sa.text("SELECT * FROM users"))
    for row in result:
        print(row.id, row.username)


    CREATE TABLE IF NOT EXISTS users (
        id INTEGER PRIMARY KEY,
        username TEXT
    )
    
INSERT INTO users (username) VALUES (?)
INSERT INTO users (username) VALUES (?)
INSERT INTO users (id, username) VALUES (?, ?)
{
  "exc_type": "IntegrityError",
  "exc_summary": "(sqlite3.IntegrityError) UNIQUE constraint failed: users.id",
  "query": "INSERT INTO users (id, username) VALUES (?, ?)",
  "params": "(1, 'Bob')"
}
SELECT * FROM users
1 Alice
2 Bob


In [29]:
import re

text = """
(sqlite3.IntegrityError) UNIQUE constraint failed: users.id
[SQL: INSERT INTO users (id, username) VALUES (?, ?)]
[parameters: (1, 'Bob')]
(Background on this error at: https://sqlalche.me/e/20/gkpj)
Traceback (most recent call last):
"""

sql_match = re.search(r"\[SQL:\s*(.*?)\]", text, re.DOTALL)
params_match = re.search(r"\[parameters:\s*(.*?)\]", text, re.DOTALL)
if sql_match:
    print(sql_match.group(1))
if params_match:
    print(params_match.group(1))

INSERT INTO users (id, username) VALUES (?, ?)
(1, 'Bob')
